In [9]:
# ============================================================
# CELL 1: IMPORTS AND CONFIG
# ============================================================
import pandas as pd
import numpy as np
import joblib
import time
import gc
from pathlib import Path

from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

LOCAL_DIR         = Path('C:\\Users\\admin\\Documents\\Glacier Project')
REGIONS           = ['R1a', 'R1b', 'R2', 'R3']
RANDOM_STATE      = 70
EASD_FEATURES     = ['elevation', 'aspect', 'slope', 'edge_distance']
PER_BUCKET_TARGET = 20000

print('Imports OK')

Imports OK


In [2]:
# ============================================================
# CELL 2: LOAD DATA AND PCA ARTIFACTS
# ============================================================
t0 = time.time()
dfs = []
for region in REGIONS:
    df = pd.read_parquet(LOCAL_DIR / f'Merged\\{region.lower()}_combined.parquet')
    df['region'] = region
    dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)
print(f'Loaded {len(df_all):,} pixels in {time.time()-t0:.1f}s')

pca    = joblib.load(LOCAL_DIR / 'Saved_Models\\pca_alphaearth.joblib')
scaler = joblib.load(LOCAL_DIR / 'Saved_Models\\scaler_alphaearth.joblib')
print(f'Loaded PCA ({pca.n_components_} components) and scaler')

Loaded 15,242,639 pixels in 15.0s
Loaded PCA (64 components) and scaler


In [3]:
# ============================================================
# CELL 3: PROJECT TO PCA SPACE (chunked to limit peak memory)
# ============================================================
ae_cols = [c for c in df_all.columns
           if c.startswith('A') and len(c) == 3 and c[1:].isdigit()]
assert len(ae_cols) == 64

# Process in 1M-row chunks so we never hold more than ~512 MB
# of temporary arrays on top of df_all at once.
CHUNK_SIZE = 1_000_000
pc_chunks = []
t0 = time.time()

for start in range(0, len(df_all), CHUNK_SIZE):
    chunk_ae = df_all.iloc[start:start + CHUNK_SIZE][ae_cols].values.astype(np.float32)
    chunk_scaled = scaler.transform(chunk_ae)
    del chunk_ae
    chunk_pca = pca.transform(chunk_scaled)
    del chunk_scaled
    pc_chunks.append(chunk_pca[:, :10].astype(np.float32))
    del chunk_pca
    gc.collect()

pc_cols = [f'PC{i+1}' for i in range(10)]
df_all[pc_cols] = np.vstack(pc_chunks)
del pc_chunks; gc.collect()

print(f'Projected to PC space in {time.time()-t0:.1f}s')


Projected to PC space in 16.4s


In [4]:
# ============================================================
# CELL 4: STRATIFIED SAMPLING (quintiles)
# df_all is deleted at the end — only df_train is kept.
# ============================================================
_, quintile_bins = pd.qcut(df_all['edge_distance'], q=5, retbins=True, duplicates='drop')
quintile_bins[0]  = -0.1
quintile_bins[-1] = np.inf

df_all['quintile'] = pd.cut(
    df_all['edge_distance'], bins=quintile_bins, labels=[1,2,3,4,5]
).astype(int)

sampled = []
for (q, m), group in df_all.groupby(['quintile', 'melt_label']):
    n_take = min(PER_BUCKET_TARGET, len(group))
    sampled.append(group.sample(n=n_take, random_state=RANDOM_STATE))
df_train = pd.concat(sampled, ignore_index=True)
del sampled, df_all; gc.collect()

print(f'Training set: {len(df_train):,} samples ({df_train["melt_label"].mean()*100:.1f}% melt)')
print(df_train.groupby(['quintile','melt_label']).size().unstack(fill_value=0))


Training set: 187,191 samples (46.6% melt)
melt_label      0      1
quintile                
1           20000  20000
2           20000  20000
3           20000  20000
4           20000  20000
5           20000   7191


In [5]:
# ============================================================
# CELL 5: FEATURE SET AND TRAIN/VAL SPLIT
# Re-run this cell to change the feature set.
# ============================================================
AE_COLS       = [c for c in df_train.columns if c.startswith('A') and len(c) == 3 and c[1:].isdigit()]
PC5           = [f'PC{i+1}' for i in range(5)]
PC10          = [f'PC{i+1}' for i in range(10)]
ELEVATION     = ['elevation']
ASPECT        = ['aspect']
SLOPE         = ['slope']
EDGE_DISTANCE = ['edge_distance']

# ── Choose which feature set to tune against ─────────────────
FEATURES     = AE_COLS + EDGE_DISTANCE + ELEVATION
VAL_FRACTION = 0.2
# Examples:
#   PC10 only     : FEATURES = PC10
#   PC10 + EASD   : FEATURES = PC10 + EASD_FEATURES
#   Raw AE        : FEATURES = AE_COLS
#   Raw AE + EASD : FEATURES = AE_COLS + EASD_FEATURES
# ─────────────────────────────────────────────────────────────

y = df_train['melt_label'].values
X = df_train[FEATURES].values.astype(np.float32)

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=VAL_FRACTION, random_state=RANDOM_STATE, stratify=y
)

print(f'Features ({len(FEATURES)}): {FEATURES[:4]} … {FEATURES[-2:]}')
print(f'Train: {len(X_tr):,}  Val: {len(X_val):,}')

Features (66): ['A00', 'A01', 'A02', 'A03'] … ['edge_distance', 'elevation']
Train: 149,752  Val: 37,439


In [13]:
# ============================================================
# CELL 6: DEFINE HYPERPARAMETER CONFIGS
# `alpha` = L2 regularisation strength (sklearn convention).
# ============================================================
"""CONFIGS = [
    {'name': 'baseline',     'hidden': (256, 128, 64),     'activation': 'relu', 'lr': 0.001,  'alpha': 0.0001, 'batch': 512},
    {'name': 'alpha_med',    'hidden': (256, 128, 64),     'activation': 'relu', 'lr': 0.001,  'alpha': 0.001,  'batch': 512},
    {'name': 'alpha_strong', 'hidden': (256, 128, 64),     'activation': 'relu', 'lr': 0.001,  'alpha': 0.01,   'batch': 512},
    {'name': 'wider',        'hidden': (512, 256, 128),    'activation': 'relu', 'lr': 0.001,  'alpha': 0.0001, 'batch': 512},
    {'name': 'wider_reg',    'hidden': (512, 256, 128),    'activation': 'relu', 'lr': 0.001,  'alpha': 0.001,  'batch': 512},
    {'name': 'wider_low_lr', 'hidden': (512, 256, 128),    'activation': 'relu', 'lr': 0.0003, 'alpha': 0.0001, 'batch': 512},
    {'name': 'lr_high',      'hidden': (256, 128, 64),     'activation': 'relu', 'lr': 0.003,  'alpha': 0.0001, 'batch': 512},
    {'name': 'lr_low',       'hidden': (256, 128, 64),     'activation': 'relu', 'lr': 0.0003, 'alpha': 0.0001, 'batch': 512},
    {'name': 'batch_small',  'hidden': (256, 128, 64),     'activation': 'relu', 'lr': 0.001,  'alpha': 0.0001, 'batch': 256},
    {'name': 'batch_large',  'hidden': (256, 128, 64),     'activation': 'relu', 'lr': 0.001,  'alpha': 0.0001, 'batch': 1024},
    {'name': 'tanh',         'hidden': (256, 128, 64),     'activation': 'tanh', 'lr': 0.001,  'alpha': 0.0001, 'batch': 512},
    {'name': 'deeper',       'hidden': (256, 128, 64, 32), 'activation': 'relu', 'lr': 0.001,  'alpha': 0.0001, 'batch': 512},
]"""

CONFIGS = [
    {'name': 'wider',    'hidden': (1024, 512, 256), 'activation': 'relu', 'lr': 0.001, 'alpha': 0.0001, 'batch': 512},
    {'name': 'wider_reg',    'hidden': (1024, 512, 256),    'activation': 'relu', 'lr': 0.001,  'alpha': 0.001,  'batch': 512},
    ]

MAX_ITER = 300  # sklearn uses early_stopping internally

print(f'{len(CONFIGS)} configs queued.\n')
print(f'  {"Name":<16} {"Architecture":<26} {"Act":<6} {"LR":<7} {"L2(alpha)":<11} {"Batch"}')
print('  ' + '-'*74)
for c in CONFIGS:
    print(f'  {c["name"]:<16} {str(c["hidden"]):<26} {c["activation"]:<6} {c["lr"]:<7} {c["alpha"]:<11} {c["batch"]}')

2 configs queued.

  Name             Architecture               Act    LR      L2(alpha)   Batch
  --------------------------------------------------------------------------
  wider            (1024, 512, 256)           relu   0.001   0.0001      512
  wider_reg        (1024, 512, 256)           relu   0.001   0.001       512


In [14]:
# ============================================================
# CELL 7: RUN ALL CONFIGS (sklearn Pipeline)
# StandardScaler + MLPClassifier. Early stopping uses sklearn's
# internal validation_fraction=0.1 so val metrics stay unbiased.
# ============================================================
def run_config(cfg):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPClassifier(
            hidden_layer_sizes  = cfg['hidden'],
            activation          = cfg['activation'],
            solver              = 'adam',
            alpha               = cfg['alpha'],
            batch_size          = cfg['batch'],
            learning_rate_init  = cfg['lr'],
            max_iter            = MAX_ITER,
            early_stopping      = True,
            validation_fraction = 0.1,
            n_iter_no_change    = 20,
            random_state        = RANDOM_STATE,
            verbose             = False,
        ))
    ])

    t0 = time.time()
    pipe.fit(X_tr, y_tr)
    elapsed = time.time() - t0

    tr_pred = pipe.predict(X_tr)
    va_pred = pipe.predict(X_val)

    def errs(y_true, y_pred):
        cm = confusion_matrix(y_true, y_pred)
        return (
            round(1 - cm[0, 0] / cm[0].sum(), 4),
            round(1 - cm[1, 1] / cm[1].sum(), 4),
            round((cm[0, 1] + cm[1, 0]) / cm.sum(), 4),
        )

    tr_ie, tr_me, tr_ae = errs(y_tr, tr_pred)
    va_ie, va_me, va_ae = errs(y_val, va_pred)
    n_iter = pipe.named_steps['mlp'].n_iter_

    return {
        'name':        cfg['name'],
        'hidden':      str(cfg['hidden']),
        'activation':  cfg['activation'],
        'lr':          cfg['lr'],
        'alpha':       cfg['alpha'],
        'batch':       cfg['batch'],
        'epochs':      n_iter,
        'time_s':      round(elapsed, 1),
        'train_ice':   tr_ie,
        'train_melt':  tr_me,
        'train_avg':   tr_ae,
        'val_ice':     va_ie,
        'val_melt':    va_me,
        'val_avg':     va_ae,
        'overfit_gap': round(va_ae - tr_ae, 4),
    }

results = []
total_start = time.time()

for i, cfg in enumerate(CONFIGS, 1):
    print(f'[{i}/{len(CONFIGS)}] {cfg["name"]}  alpha={cfg["alpha"]}  batch={cfg["batch"]}... ',
          end='', flush=True)
    try:
        row = run_config(cfg)
        results.append(row)
        print(f'{row["time_s"]:>6.1f}s  val_avg={row["val_avg"]:.4f}  gap={row["overfit_gap"]:+.4f}')
    except Exception as e:
        print(f'FAILED: {e}')

print(f'\nFinished {len(results)}/{len(CONFIGS)} configs in {(time.time()-total_start)/60:.1f} min')

[1/2] wider  alpha=0.0001  batch=512...  873.2s  val_avg=0.1327  gap=+0.1104
[2/2] wider_reg  alpha=0.001  batch=512... 1123.3s  val_avg=0.1315  gap=+0.0783

Finished 2/2 configs in 33.4 min


In [15]:
# ============================================================
# CELL 8: RESULTS TABLE
# Sorted by val_avg (best first). Re-run any time to refresh.
# ============================================================
df_results = pd.DataFrame(results).sort_values('val_avg').reset_index(drop=True)
df_results.index += 1  # rank from 1

print(f'Features: {FEATURES}')
print(f'Ranked by val avg error ({len(df_results)} configs)\n')

display_cols = [
    'name', 'hidden', 'activation', 'lr', 'alpha', 'batch', 'epochs', 'time_s',
    'train_avg', 'val_ice', 'val_melt', 'val_avg', 'overfit_gap',
]
pd.set_option('display.max_colwidth', 24)
pd.set_option('display.width', 160)
print(df_results[display_cols].to_string())

print(f'\nBest config: {df_results.iloc[0]["name"]}  val_avg={df_results.iloc[0]["val_avg"]:.4f}')

Features: ['A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45', 'A46', 'A47', 'A48', 'A49', 'A50', 'A51', 'A52', 'A53', 'A54', 'A55', 'A56', 'A57', 'A58', 'A59', 'A60', 'A61', 'A62', 'A63', 'edge_distance', 'elevation']
Ranked by val avg error (2 configs)

        name            hidden activation     lr   alpha  batch  epochs  time_s  train_avg  val_ice  val_melt  val_avg  overfit_gap
1  wider_reg  (1024, 512, 256)       relu  0.001  0.0010    512      41  1123.3     0.0532   0.1308    0.1325   0.1315       0.0783
2      wider  (1024, 512, 256)       relu  0.001  0.0001    512      65   873.2     0.0223   0.1299    0.1358   0.1327       0.1104

Best config: wider_reg  val_avg=0.1315


In [9]:
# ============================================================
# CELL 9: SAVE / LOAD RESULTS  (optional)
# ============================================================
RESULTS_FILE = LOCAL_DIR / 'mlp_tuning_results.csv'

# Save
df_results.to_csv(RESULTS_FILE, index=True)
print(f'Saved to {RESULTS_FILE}')

# Load previous results and merge (uncomment to use)
# df_prev = pd.read_csv(RESULTS_FILE, index_col=0)
# df_results = pd.concat([df_prev, df_results]).drop_duplicates('name').sort_values('val_avg').reset_index(drop=True)
# df_results.index += 1

Saved to C:\Users\admin\Documents\Glacier Project\mlp_tuning_results.csv
